# FP-Growth Dashboard Visualization - Cohort Runner

This notebook runs **FP-Growth analysis** for all configured cohorts and age bands.

- **Script**: `9_dashboard_visuals/fpgrowth/create_fpgrowth_visuals.py`
- **Purpose**: Extract frequent itemsets and association rules for dashboard visualization
- **Outputs**: Itemsets, rules, and network visualizations (NOT model features - visualization only)

## Features

✅ **Dynamic cohort selection** - Configure which cohorts/age bands to run  
✅ **Idempotent** - Automatically skips completed analyses  
✅ **Parallel processing** - Leverages all available CPU cores  
✅ **S3 integration** - Results automatically synced to S3

## Usage

1. Configure cohorts and age bands in the configuration cell below
2. Run individual cells for specific cohorts, or use "Run All" for batch processing
3. Results are saved locally and synced to S3 automatically

In [ ]:
# Environment Setup

import os
import sys
from pathlib import Path

# Resolve Python binary
def resolve_python_bin() -> Path:
    env_bin = os.environ.get("COHORT_RUNNER_PYTHON")
    if env_bin:
        return Path(env_bin)
    return Path(sys.executable)

PYTHON_BIN = resolve_python_bin()
print(f"[INFO] Using Python binary: {PYTHON_BIN}")

# Resolve project root
def resolve_project_root() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parents[1]
    notebook_path = Path(os.getcwd()).resolve()
    if notebook_path.name == "fpgrowth" and "9_dashboard_visuals" in str(notebook_path.parent):
        return notebook_path.parent.parent
    for parent in notebook_path.parents:
        if parent.name == "pgx-analysis":
            return parent
    return notebook_path

PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import constants for available cohorts and age bands
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

## Configuration

Select which cohorts and age bands to process. Leave empty lists to process all.

In [ ]:
# Configuration: Select cohorts and age bands to process
# Leave as empty lists [] to process all available

COHORTS_TO_RUN = []  # e.g., ['opioid_ed'] or [] for all
AGE_BANDS_TO_RUN = []  # e.g., ['0-12', '13-24'] or [] for all

# If empty, use all available
if not COHORTS_TO_RUN:
    COHORTS_TO_RUN = COHORT_NAMES.copy()
if not AGE_BANDS_TO_RUN:
    AGE_BANDS_TO_RUN = AGE_BANDS.copy()

print(f"Will process {len(COHORTS_TO_RUN)} cohort(s) and {len(AGE_BANDS_TO_RUN)} age band(s)")
print(f"Cohorts: {COHORTS_TO_RUN}")
print(f"Age bands: {AGE_BANDS_TO_RUN}")

## Run FP-Growth Analysis

Each cell below runs FP-Growth for a specific cohort/age band combination.

In [ ]:
# Run FP-Growth for all configured cohort/age band combinations

import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

FAIL_FAST = True  # Stop on first failure; set to False to continue on errors

# Generate all combinations
combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Running FP-Growth analysis for {len(combinations)} cohort/age band combinations...")
print("=" * 80)

for cohort_name, age_band in combinations:
    print(f"\n[FP-Growth] Starting: {cohort_name} / {age_band}")
    print("-" * 80)
    
    result = subprocess.run(
        [str(PYTHON_BIN), "9_dashboard_visuals/fpgrowth/create_fpgrowth_visuals.py",
         "--cohort-name", cohort_name,
         "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    
    if result.returncode == 0:
        print(f"[FP-Growth] COMPLETED: {cohort_name} / {age_band}")
    else:
        msg = f"[FP-Growth] FAILED ({result.returncode}): {cohort_name} / {age_band}"
        print(msg)
        if FAIL_FAST:
            raise RuntimeError(msg)

print("\n" + "=" * 80)
print("All FP-Growth analyses completed (or were skipped as already done).")
print("=" * 80)